In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DecimalType
from delta.tables import DeltaTable

def remove_duplicates(df, primary_key):
    return df.dropDuplicates([primary_key])

def remove_nulls(df, critical_columns):
    return df.dropna(subset=critical_columns)

def fill_nulls(df, default_values):
    return df.fillna(default_values)

def load_data(format_type, file_name):
    base_path = "abfss://fabricws@onelake.dfs.fabric.microsoft.com/lh_fabric.Lakehouse/Files/bankdata-dim"
    return (spark.read.format(format_type)
            .option("header", "true")
            .option("inferSchema", "true")
            .load(f"{base_path}/{file_name}.csv"))

def apply_schema_and_casting(df, casting_rules):
    """
    Dynamically applies casting, trimming, upper-casing, AND renames 
    the column from source name to target name.
    """
    select_exprs = []
    for target_col, rule in casting_rules.items():
        # Look for the column using its source name, default to target name if not provided
        source_col = rule.get("source_col_name", target_col)
        expr = F.col(source_col)
        
        # Apply string cleanups if specified
        if rule.get("trim"):
            expr = F.trim(expr)
        if rule.get("upper"):
            expr = F.upper(expr)
            
        # Apply data type casting
        if "type" in rule:
            if rule["type"] == "date" and "format" in rule:
                expr = F.to_date(expr, rule["format"])
            else:
                expr = expr.cast(rule["type"])
                
        # ALIAS directly to the target table's column name
        select_exprs.append(expr.alias(target_col))
        
    return df.select(*select_exprs)


def execute_scd1_merge(df_source, target_table_name, primary_key):
    if spark.catalog.tableExists(target_table_name):
        print(f"Target table '{target_table_name}' exists. Performing Upsert...")
        target_delta = DeltaTable.forName(spark, target_table_name)
        
        all_columns = df_source.columns
        non_key_columns = [col for col in all_columns if col != primary_key]
        
        update_map = {col: f"source.{col}" for col in non_key_columns}
        insert_map = {col: f"source.{col}" for col in all_columns}
        
        (target_delta.alias("target")
         .merge(df_source.alias("source"), f"target.{primary_key} = source.{primary_key}")
         .whenMatchedUpdate(set=update_map)
         .whenNotMatchedInsert(values=insert_map)
         .execute())
    else:
        print(f"Target table '{target_table_name}' does not exist. Initializing...")
        df_source.write.format("delta").mode("overwrite").saveAsTable(target_table_name)
        
    print(f"Successfully processed {target_table_name}.\n" + "="*40)

# ====================================================================
# CONFIGURATION WITH SOURCE-TO-TARGET MAPPING
# ====================================================================

pipeline_config = {
    "dim_customer": {
        "primary_key": "customer_key", 
        "critical_columns": ["customer_id", "full_name"],
        "default_values": {"kyc_status": "UNKNOWN","customer_segment":"UNKNOWN"},
        # Maps target rules. Added "source_col_name" to map the raw CSV header to the target.
        "schema_casting": {
            "customer_key": {"source_col_name": "customer_key", "type": IntegerType()},
            "customer_id": {"source_col_name": "customer_id", "type": StringType(), "trim": True},
            "full_name": {"source_col_name": "full_name", "type": StringType() , "trim": True},
            "gender": {"source_col_name": "gender", "type": StringType(),"trim": True},
            "date_of_birth": {"source_col_name": "date_of_birth", "type": "date", "format": "yyyy-MM-dd"},
            "city": {"source_col_name": "city", "type": StringType(),"trim": True},
            "state": {"source_col_name": "state", "type": StringType(),"trim": True},
            "customer_segment": {"source_col_name": "customer_segment", "type": StringType(),"trim": True},
            "kyc_status": {"source_col_name": "kyc_status", "type": StringType(),"trim": True, "upper": True},
            "onboard_date": {"source_col_name": "onboard_date", "type": "date", "format": "yyyy-MM-dd"},
        }
    },
        "dim_product": {
        "primary_key": "product_key", 
        "critical_columns": ["product_key", "product_code"],
        "default_values" : {"product_name": "UNKNOWN PRODUCT",
            "product_category": "STANDARD",
            "interest_rate_percent": 0.00,
            "fee_percent": 0.00,
            "is_active": 'U'
        },
        # Maps target rules. Added "source_col_name" to map the raw CSV header to the target.
        "schema_casting": {
            "product_key": {"source_col_name": "product_key", "type": IntegerType()},
            "product_code": {"source_col_name": "product_code", "type": StringType(), "trim": True},
            "product_name": {"source_col_name": "product_name", "type": StringType() , "trim": True},
            "product_category": {"source_col_name": "product_category", "type": StringType(),"trim": True},
            "interest_rate_percent": {"source_col_name": "interest_rate_percent", "type": DecimalType(5,2)},
            "fee_percent": {"source_col_name": "fee_percent", "type": DecimalType(5,2)},
            "is_active": {"source_col_name": "is_active", "type": StringType(),"trim": True}
        }
    },
        "dim_branch": {
        "primary_key": "branch_key", 
        "critical_columns": ["branch_key", "branch_code"],
        "default_values" :{
            "branch_name": "UNKNOWN BRANCH",
            "city": "UNKNOWN",
            "state": "UNKNOWN",
            "country": "UNKNOWN",
            "region": "UNKNOWN",
            "branch_type": "STANDARD"
        },
        # Maps target rules. Added "source_col_name" to map the raw CSV header to the target.
        "schema_casting": {
            "branch_key": {"source_col_name": "branch_key", "type": IntegerType()},
            "branch_code": {"source_col_name": "branch_code", "type": StringType(), "trim": True},
            "branch_name": {"source_col_name": "branch_name", "type": StringType() , "trim": True},
            "city": {"source_col_name": "city", "type": StringType(),"trim": True},
            "state": {"source_col_name": "state", "type": StringType(),"trim": True},
            "country": {"source_col_name": "country", "type": StringType(),"trim": True},
            "region": {"source_col_name": "region", "type": StringType(),"trim": True},
            "open_date": {"source_col_name": "open_date", "type": "date", "format": "M/d/yyyy"},
            "branch_type": {"source_col_name": "branch_type", "type": StringType(),"trim": True},
        }
    },
    "dim_account": {
        "primary_key": "account_key", 
        "critical_columns": ["account_number", "customer_key", "branch_key", "product_key"],
        "default_values" :{"account_status": "UNKNOWN", "currency": "INR", "current_balance": 0.0},
        # Maps target rules. Added "source_col_name" to map the raw CSV header to the target.
        "schema_casting": {
            "account_key": {"source_col_name": "account_key", "type": IntegerType()},
            "account_number": {"source_col_name": "account_number", "type": StringType(), "trim": True},
            "customer_key": {"source_col_name": "customer_key", "type": IntegerType()},
            "branch_key": {"source_col_name": "branch_key", "type": IntegerType()},
            "product_key": {"source_col_name": "product_key", "type": IntegerType()},
            "account_open_date": {"source_col_name": "account_open_date", "type": "date", "format": "yyyy-MM-dd"},
            "account_status": {"source_col_name": "account_status", "type": StringType(), "trim": True},
            "currency": {"source_col_name": "currency", "type": StringType(), "trim": True},
            "current_balance": {"source_col_name": "current_balance", "type": DecimalType(9,2)}
        }
    }
}

# ====================================================================
# 4. PIPELINE EXECUTION LOOP
# ====================================================================

for table_name, config in pipeline_config.items():
    print(f"Starting processing for table: {table_name}")
    
    # Load raw data (columns are still named 'SRC_CUST_KEY', 'CUST_ID_NUM', etc.)
    df_raw = load_data("csv", table_name)
    
    # Step 1: Enforce Schema & Rename Columns immediately to target names
    # Now df_transformed columns are 'customer_key', 'customer_id', etc.
    df_transformed = apply_schema_and_casting(df_raw, config["schema_casting"])
    
    # Step 2: Clean and Deduplicate using unified target column names
    df_deduped = remove_duplicates(df_transformed, config["primary_key"])
    df_clean_keys = remove_nulls(df_deduped, config["critical_columns"])
    df_no_nulls = fill_nulls(df_clean_keys, config["default_values"])
    
    # Step 3: Execute Merge safely
    execute_scd1_merge(df_no_nulls, table_name, config["primary_key"])

StatementMeta(, 006dc458-0093-4b40-ada1-817619ce5b70, 3, Finished, Available, Finished, False)

Starting processing for table: dim_customer
Target table 'dim_customer' does not exist. Initializing...
Successfully processed dim_customer.
Starting processing for table: dim_product
Target table 'dim_product' does not exist. Initializing...
Successfully processed dim_product.
Starting processing for table: dim_branch
Target table 'dim_branch' does not exist. Initializing...
Successfully processed dim_branch.
Starting processing for table: dim_account
Target table 'dim_account' does not exist. Initializing...
Successfully processed dim_account.
